# 🛠️ Adım 2: Gelişmiş Özellik Mühendisliği (Feature Engineering)
Bu notebook, ham veriden makine öğrenmesine girdi olacak 10 adet kompleks özelliğin üretim sürecini kapsar.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("FeatureEngFull").getOrCreate()

### 1. Veri Hazırlama

In [ ]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("/app/dataset/bankdataset.csv")
df = df.withColumnRenamed("Value", "value").withColumnRenamed("Transaction_count", "transaction_count").withColumnRenamed("Domain", "domain").withColumnRenamed("Location", "location")

### 2. Temel ve Tarihsel Özellikler

In [ ]:
# Islem basina deger
df = df.withColumn("value_per_transaction", round(col("value") / col("transaction_count"), 2))

# Tarih bilesenleri
df = df.withColumn("parsed_date", to_date(col("Date"), "yyyy-MM-dd"))
df = df.withColumn("month", month("parsed_date")) \
       .withColumn("day_of_week", dayofweek("parsed_date")) \
       .withColumn("is_weekend", when((col("day_of_week") == 1) | (col("day_of_week") == 7), 1).otherwise(0))

### 3. Kategorik ve İstatistiksel Özellikler (Advanced)

In [ ]:
# Value Category (Kategorizasyon)
df = df.withColumn("value_category", when(col("value") < 400000, 0).when(col("value") < 700000, 1).otherwise(2))

# Lokasyon Bazli Ortalama (Window Function)
loc_window = Window.partitionBy("location")
df = df.withColumn("location_avg_value", round(avg("value").over(loc_window), 2))

# Z-Score (Global Normalizasyon)
stats = df.select(avg("value").alias("m"), stddev("value").alias("s")).collect()[0]
df = df.withColumn("value_zscore", round((col("value") - stats["m"]) / stats["s"], 4))

df.select("value", "value_per_transaction", "is_weekend", "location_avg_value", "value_zscore").show(5)

### 4. Sonuçların Kaydedilmesi

In [ ]:
print("Özellik mühendisliği tamamlandı. Toplam Ozellik Sayisi: 10")
# df.write.format("delta").mode("overwrite").save("/app/delta-lake/gold/features")